# Synthetic Augmentation (Type V Focus)


In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

DATA_ROOT = '/mounts/mecd-ap-g5/data'
RESULTS_DIR = '/mounts/mecd-ap-g5/results/synthetic_augmentation'
os.makedirs(RESULTS_DIR, exist_ok=True)
IMG_SIZE = 320
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load metadata
meta = pd.read_csv(os.path.join(DATA_ROOT, 'MIQR-CC-Dataset', 'metadata.csv'))
EXCLUDED_LABELS = {'unlabeled', 'unlabelled', 'unlabbeled'}
filtered = meta[(meta['Keep'].astype(str).str.strip().str.lower() == 'keep') & (~meta['Label'].astype(str).str.strip().str.lower().isin(EXCLUDED_LABELS))].copy()
if 'image_type' not in filtered.columns:
    raise ValueError('metadata.csv missing image_type column')
filtered['image_path'] = filtered['processed_image_path'].apply(lambda p: os.path.join(DATA_ROOT, 'MIQR-CC-Dataset', p))
filtered = filtered[filtered['image_path'].apply(os.path.exists)].copy()

# Focus on type V
v_df = filtered[filtered['image_type'].astype(str).str.upper() == 'V'].copy().reset_index(drop=True)
print('Type V samples:', len(v_df))
display(v_df[['processed_image_path', 'Label', 'image_type']].head())


Type V samples: 47


,processed_image_path,Label,image_type
0,processed/1487_image18828_section_2.png,Biliary Leaks,V
1,processed/1488_image18832_section_1.png,Normal,V
2,processed/1488_image18832_section_2.png,Normal,V
3,processed/1494_image18843_section_3.png,Normal,V
4,processed/1498_image18848_section_1.png,Normal,V


In [2]:
# Strong but realistic augmentation pipeline
aug = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.92, 1.08)),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

def save_synthetic_variants(row, n_variants=10):
    src_path = row['image_path']
    label = row['Label']
    patient_id = row['patient_id']
    out_dir = os.path.join(RESULTS_DIR, 'images', label)
    os.makedirs(out_dir, exist_ok=True)
    img = Image.open(src_path).convert('RGB')
    saved = []
    for i in range(n_variants):
        aug_img = aug(img)
        fname = f'synthetic_pid{patient_id}_idx{row.name}_v{i}.png'
        out_path = os.path.join(out_dir, fname)
        aug_img.save(out_path)
        saved.append(out_path)
    return saved

# Generate synthetic samples
N_VARIANTS_PER_IMAGE = 12
records = []
for _, row in v_df.iterrows():
    paths = save_synthetic_variants(row, n_variants=N_VARIANTS_PER_IMAGE)
    for p in paths:
        records.append({
            'synthetic_path': p,
            'Label': row['Label'],
            'image_type': 'V',
            'source_patient_id': row['patient_id'],
            'source_image_path': row['image_path'],
        })

synth_df = pd.DataFrame(records)
synth_csv = os.path.join(RESULTS_DIR, 'synthetic_index.csv')
synth_df.to_csv(synth_csv, index=False)
print('Saved synthetic images:', len(synth_df))
print('Index CSV:', synth_csv)
display(synth_df.head())

Saved synthetic images: 564
Index CSV: /mounts/mecd-ap-g5/results/synthetic_augmentation/synthetic_index.csv


,synthetic_path,Label,image_type,source_patient_id,source_image_path
0,/mounts/mecd-ap-g5/results/synthetic_augmentat...,Biliary Leaks,V,1487,/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/proces...
1,/mounts/mecd-ap-g5/results/synthetic_augmentat...,Biliary Leaks,V,1487,/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/proces...
2,/mounts/mecd-ap-g5/results/synthetic_augmentat...,Biliary Leaks,V,1487,/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/proces...
3,/mounts/mecd-ap-g5/results/synthetic_augmentat...,Biliary Leaks,V,1487,/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/proces...
4,/mounts/mecd-ap-g5/results/synthetic_augmentat...,Biliary Leaks,V,1487,/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/proces...
